In [ ]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "15"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    print(f"{e}: 로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-rag"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=False)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# 기타 환경 설정
import torch
import matplotlib.pyplot as plt
import gc


# 시각화 관련 설정
if not IS_COLAB_MODE:
    import matplotlib.font_manager as fm
    try:
        plt.rcParams['font.family'] = 'Apple SD Gothic Neo'
    except:
        try:
            plt.rcParams['font.family'] = 'NanumGothic'
        except:
            plt.rcParams['font.family'] = 'AppleGothic'

    plt.rcParams['axes.unicode_minus'] = False
    fm._load_fontmanager(try_read_cache=False)


# 디바이스 설정
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device("mps") # 맥 GPU
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda") # 윈도우 GPU
else:
    DEVICE = torch.device("cpu") # CPU


# 캐시 지우기 함수 생성
def clean_cache():
    gc.collect()
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()


# MallocStackLogging 에러 출력 방지
os.environ.pop("MallocStackLogging", None)
os.environ.pop("MallocStackLoggingNoCompact", None)
os.environ.pop("DYLD_INSERT_LIBRARIES", None)

In [ ]:
# 학습 유틸리티 설정

# wandb
if IS_COLAB_MODE:
    subprocess.run(["pip", "install", "wandb", "-qU"])

import wandb

wandb.login(key=get_secret("WANDB_API_KEY"))


# 텔레그램
def send_telegram_msg(message):
    token = get_secret("TELEGRAM_BOT_TOKEN")
    chat_id = get_secret("TELEGRAM_CHAT_ID")

    url = f'https://api.telegram.org/bot{token}/sendMessage'
    payload = {
        'chat_id': chat_id,
        'text': message
    }

    requests.post(url, json=payload)

In [ ]:
resume = "must"

wandb.init(
    project=PROJECT_NUM,
    name="SEONGIL WON",
    entity="wsi0720-sungkyunkwan-university",
    resume=resume,
    dir=SAVE_DIR,
    id=id,
    config={"model": MODEL_NAME}
)

wandb.log({
    "epoch": epoch,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "best_val_loss": self.best_loss
})


In [ ]:
id_path = os.path.join(simple_trainer_config.save_dir, f"wandb_id_{simple_trainer_config.mode}.txt")

if not os.path.exists(simple_trainer_config.save_dir):

    os.mkdir(simple_trainer_config.save_dir)

    id = wandb.util.generate_id()
    with open(id_path, "w") as f:
        f.write(id)
    resume = "allow"

else:
    if os.path.exists(id_path):
        with open(id_path, "r") as f:
            id = f.read().strip()
        resume = "must"

    else:
        id = wandb.util.generate_id()
        with open(id_path, "w") as f:
            f.write(id)
        resume = "allow"

wandb.init(
    project="13",
    name="SEONGIL WON",
    entity="wsi0720-sungkyunkwan-university",
    resume=resume,
    dir=SAVE_DIR,
    id=id,
    config={
        "model": model_name
        }
    )


resume_dir = os.path.join(simple_trainer_config.save_dir, "last_model")

if os.path.exists(resume_dir):
    model = AutoModelForSequenceClassification.from_pretrained(resume_dir)
    tokenizer = AutoTokenizer.from_pretrained(resume_dir)
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)


checkpoint_path = os.path.join(resume_dir, "training_state.pt")

if not os.path.exists(checkpoint_path):
    start_epoch = 0
else:
    checkpoint = torch.load(checkpoint_path)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = checkpoint["epoch"] + 1